<a href="https://colab.research.google.com/github/KyraWolfe/Into-to-Comp-Sys-Final-Project/blob/main/compSysFinalProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#install kaggle and tensorflow
!pip install kaggle --quiet
!pip install -q transformers datasets seaborn scikit-learn

In [ ]:
#upload kaggle json file
from google.colab import files
files.upload()

In [ ]:
#create directory for kaggle
#~/.kaggle is the name i think
!mkdir ~/.kaggle

In [ ]:
#copy the json into dir
!cp kaggle.json ~/.kaggle/

In [ ]:
#permissions
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
#finding dataset
!kaggle datasets list -s 'CEAS_08.csv'

In [ ]:
#downloading dataset
!kaggle datasets download -d 'naserabdullahalam/phishing-email-dataset' -p ../data/phishingEmailsFinalProject

In [ ]:
#unzipping the files
!unzip ../data/phishingEmailsFinalProject/phishing-email-dataset.zip -d ../data/phishingEmailsFinalProject

In [ ]:
#more permissions
!chmod 777 ../data/phishingEmailsFinalProject/CEAS_08.csv

In [ ]:
#loading database
import pandas as pd
import re
import torch

from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments
from transformers import DataCollatorWithPadding
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
df = pd.read_csv('../data/phishingEmailsFinalProject/CEAS_08.csv')
df.head()

In [ ]:
# data info
df.info()

In [ ]:
#label contains 1 or 0, 1 means scam and 0 means safe
#goal is 0.5 error, 0.5 confidence
#features on sender, subject, body, url
#VC would be 5

In [ ]:
sdf=df.sample(n=15000, random_state=42).reset_index(drop=True)
print(sdf.head())

In [ ]:
sdf.rename(columns={'text_combined': 'text'}, inplace=True)
sdf['label'] = sdf['label'].astype(int)

#remove the empty slots
sdf.dropna(inplace=True)

# makes sure all text is lowercase to run through BERT easier
def clean_text(text):
  return text.lower()
sdf['text'] = sdf['text'].apply(clean_text)

In [ ]:
sdf['special_chars'] = sample_data/re.sdf['text'].apply(lambda x: sum(not c.isalnum() and not c.isspace() for c in x))

In [ ]:
from collections import Counter
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')


stopw = set(stopwords.words('english'))
phishw = ' '.join(sdf[sdf['label']==1]['text']).lower().split()
filterw = [word for word in phishw if word.isalpha() and word not in stopw]
freqw = Counter(filterw).most_common(30)

In [ ]:
train_text, test_text, train_label, test_label = train_test_split(sdf['text'].tolist(), sdf['label'].tolist(), test_size=0.2, random_state=42, stratify=sdf['label'])
trainT, valT, trainL, valL, = train_test_split(train_text, train_label, test_size=0.125, random_state=42, stratify=train_label)

In [ ]:
token = BertTokenizer.from_pretrained("bert-base-uncased")
train_encode = token(trainT, truncation=True, padding=True)
val_encode = token(valT, truncation=True, padding=True)

In [ ]:
class PhishData(torch.utils.data.Dataset):
    def __init__(self, encode, labels):
        self.encode = encode
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encode.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

trainData = PhishData(train_encode, trainL)
valData = PhishData(val_encode, valL)

In [ ]:
PhishModel = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)